# Analysis of Results



In [1]:
import pandas as pd
import json
from pathlib import Path
from dataclasses import dataclass
import matplotlib.pyplot as plt
import numpy as np
import csv
from dotenv import load_dotenv
import os
import qvarnet
load_dotenv()  # Load environment variables from .env file
print(os.getcwd())
if os.getenv('MOUNTPOINT') is None:
    raise ValueError("MOUNTPOINT environment variable is not set.")
import utils

/home/pau/PhD/data_analysis


## Change to the directory where the results are stored

In [2]:
os.chdir(os.getenv('MOUNTPOINT'))

## Data loading and processing

### Loading results in a dataframe

In [8]:
PARQUET_DIR = "/home/pau/PhD/data_analysis"
if "results.parquet" not in os.listdir(PARQUET_DIR):
    df = utils.load_simulation_results('./out/20260211_142926')
    df.to_parquet(WHERE_TO_SAVE+"/results.parquet", engine="fastparquet")
else:
    df = pd.read_parquet(PARQUET_DIR+"/results.parquet", engine="fastparquet")
print(df.head())

   seed  total_energy       std  training_time_seconds  best_score  \
0    42     21.395267  3.152891              38.339563   27.701050   
1    42      5.650299  1.682009              27.508885    9.014318   
2    42      4.518309  1.176057              16.415093    6.870423   
3    42      4.970542  1.889289              18.872814    8.749120   
4    42     25.625671  2.646152             219.222425   30.917974   

                                                path  \
0  out/20260211_142926/config_8_hu_3_l_50_particl...   
1  out/20260211_142926/config_4_hu_3_l_10_particl...   
2  out/20260211_142926/config_4_hu_1_l_10_particl...   
3  out/20260211_142926/config_8_hu_1_l_10_particl...   
4  out/20260211_142926/config_32_hu_1_l_200_parti...   

                     model_type model_architecture model_activation  \
0  exponential-mlp-fourth-decay   [50, 8, 8, 8, 1]             tanh   
1  exponential-mlp-fourth-decay   [10, 4, 4, 4, 1]             tanh   
2  exponential-mlp-fourth-dec

### Cleaning and transforming the data

In [9]:
df = utils.separate_input_dim_and_hidden_dim(df)
df = utils.add_energy_per_particle(df)
df = utils.clean_non_informative_columns(df)

Column 'seed' has only 1 unique group size, dropping.
Column 'model_type' has only 1 unique group size, dropping.
Column 'model_activation' has only 1 unique group size, dropping.
Column 'training_num_epochs' has only 1 unique group size, dropping.
Column 'sampler_type' has only 1 unique group size, dropping.
Column 'sampler_PBC' has only 1 unique group size, dropping.
Column 'optimizer_type' has only 1 unique group size, dropping.


In [10]:
df.head()

,total_energy,std,training_time_seconds,best_score,path,training_batch_size,sampler_step_size,sampler_chain_length,optimizer_learning_rate,input_dim,hidden_dim,energy_per_particle,std_energy_per_particle,best_score_per_particle
0,21.395267,3.152891,38.339563,27.701050,out/20260211_142926/config_8_hu_3_l_50_particl...,500,0.05,100,0.001,50,"[8, 8, 8]",0.427905,0.063058,0.554021
1,5.650299,1.682009,27.508885,9.014318,out/20260211_142926/config_4_hu_3_l_10_particl...,1000,0.50,125,0.001,10,"[4, 4, 4]",0.565030,0.168201,0.901432
2,4.518309,1.176057,16.415093,6.870423,out/20260211_142926/config_4_hu_1_l_10_particl...,500,0.05,125,0.010,10,[4],0.451831,0.117606,0.687042
3,4.970542,1.889289,18.872814,8.749120,out/20260211_142926/config_8_hu_1_l_10_particl...,1000,0.05,150,0.001,10,[8],0.497054,0.188929,0.874912
4,25.625671,2.646152,219.222425,30.917974,out/20260211_142926/config_32_hu_1_l_200_parti...,500,0.05,150,0.001,200,[32],0.128128,0.013231,0.154590


## Data visualization and analysis

In [ ]:
test_row = utils.localize_element_in_dataframe(df, 10, "[64, 64]", 0.01, 150, 0.5)
current_path = test_row['path']
print(current_path)

In [ ]:
energy_history = utils.get_energy_history(current_path.iloc[0])
plt.plot(energy_history)

In [ ]:
model, jax_params, input_dim = utils.load_model_from_path(current_path.iloc[0])
print(model)

In [ ]:
utils.plot_wavefunction_one_particle(model, jax_params, input_dim)

In [ ]:
test_row.iloc[0]